# Phase 9B — CIDACC Classical Computer Vision Baseline

Reproducible notebook for the classical (non-deep-learning) cell detection/counting baseline on the CIDACC dataset.

- Raw data: `data/raw/cidac/` (read-only; never modified)
- Script: `src/vision/cidacc_classical_baseline.py`
- Outputs: `data/processed/cidac/`, `reports/phase9b_cidacc_classical_baseline.md`

**Label policy:** class 0 = individual *C. vulgaris* cell (the only counted class); class 1 = cluster/noise object (never counted as one cell).

**Excluded from evaluation:** `distinct/335`, `distinct/515` (empty bbox files whose masks contain objects). The 13 blank-mask samples are retained for box detection; their masks are never used as ground truth (masks are not used at all in Phase 9B).

No neural network is trained here.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'vision'))

import cidacc_classical_baseline as cb

OUT = PROJECT_ROOT / 'data' / 'processed' / 'cidac'
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

## 1. Analysis requirements
Usable images, class totals, distributions, intensity/contrast, feasibility.

In [ ]:
analysis = json.loads((OUT / 'phase9b_analysis.json').read_text())
for k in ('usable_images', 'excluded_images', 'class0_annotations',
          'class1_annotations', 'images_with_zero_class0', 'images_with_class1'):
    print(f'{k}: {analysis[k]}')
print('\nclass-0 per image:', analysis['class0_per_image'])
print('class-1 per image:', analysis['class1_per_image'])
print('by subset:', json.dumps(analysis['by_subset'], indent=2))
print('intensity:', analysis['intensity'])
print('cell vs background delta:', analysis['cell_vs_background_delta'])
print('gt box area:', analysis['gt_box_area_px'])

**Feasibility:** cells are consistently darker than background (median delta ~ -14 gray levels), small and compact (class-0 box area median ~ 268 px2), so classical threshold/connected-component segmentation is feasible in principle. Two intensity regimes exist (~7% bright images). The hemocytometer grid, debris, merged touching cells and clusters overlapping cells are the expected failure sources.

## 2. Image-level split
Deterministic (seed 42), stratified by subset x class-1 presence; each complete image in exactly one split; excluded images in no split. No cell-level or patch-level splitting. Limitation: no acquisition/session metadata exists for the 628 images, so visually related frames cannot be grouped.

In [ ]:
manifest = pd.read_csv(OUT / 'phase9b_split_manifest.csv', dtype={'image_id': str})
meta = json.loads((OUT / 'phase9b_split_metadata.json').read_text())
print(meta)
usable = manifest[~manifest['excluded']]
print('\nsplit sizes:\n', usable['split'].value_counts())
print('\nstrata (subset x has_c1) x split:\n',
      usable.assign(has_c1=usable['n_c1'] > 0)
            .groupby(['subset', 'has_c1', 'split']).size().unstack(fill_value=0))
usable.groupby(['subset', 'split'])[['n_c0', 'n_c1']].sum()

## 3. Method comparison (validation selection -> test evaluation)
Three classical detectors: `otsu` (global threshold), `tophat` (black top-hat), `log` (LoG blob response). Selection by validation count MAE; test reported once.

In [ ]:
selection = json.loads((OUT / 'phase9b_selection.json').read_text())
grid = pd.DataFrame(selection['grid_results'])
grid[['method', 'config_idx', 'val_mae', 'val_rmse', 'val_bias', 'val_f1']].sort_values('val_mae')

In [ ]:
results = json.loads((OUT / 'phase9b_results_summary.json').read_text())
rows = []
for method, res in results['test_metrics'].items():
    c, d = res['count'], res['detection']
    rows.append({'method': method, 'mae': c['mae'], 'rmse': c['rmse'],
                 'bias': c['bias'], 'mape_gt>0': c.get('mape_pct_gt_gt0'),
                 'wape': c.get('wape_pct'),
                 'precision': d['precision'], 'recall': d['recall'],
                 'f1': d['f1'], 'mean_iou': d['mean_iou_matched']})
test_table = pd.DataFrame(rows).set_index('method')
print('selected method:', results['selection']['best_method'],
      results['selection']['best_params'])
test_table

## 4. Per-image errors of the selected method on test

In [ ]:
best = results['selection']['best_method']
per_image = pd.read_csv(OUT / f'phase9b_test_per_image_{best}.csv', dtype={'image_id': str})
print('images:', len(per_image),
      '| |err| p50/p90/max:',
      per_image['err'].abs().quantile([0.5, 0.9, 1.0]).tolist())
per_image.sort_values('err', key=abs, ascending=False).head(10)

## 5. Reproduce one image end-to-end with the script functions

In [ ]:
row = per_image.sort_values('err', key=abs, ascending=False).iloc[0]
subset, image_id = row['subset'], str(row['image_id'])
gray = cb.load_gray(subset, image_id)
gt = cb.load_gt_boxes(subset, image_id)
cells, clusters = cb.DETECTORS[best](gray, dict(results['selection']['best_params']))
tp, fp, fn, ious = cb.match_boxes(cells, gt[0])
print(f"{subset}/{image_id}: gt_c0={len(gt[0])} gt_c1={len(gt[1])} "
      f"pred_cells={len(cells)} pred_clusters={len(clusters)} "
      f"tp={tp} fp={fp} fn={fn} "
      f"mean_iou={sum(ious)/len(ious) if ious else float('nan'):.3f}")

## 6. Qualitative overlays
Saved in `data/processed/cidac/qualitative/` (green = GT class 0, yellow = GT class 1, blue = classical prediction).

In [ ]:
from IPython.display import Image, display
for p in sorted((OUT / 'qualitative').glob('*.png')):
    display(Image(filename=str(p), width=320))

## 7. Conclusion
- The classical baseline provides a reference point but its counting error and detection F1 are far from annotation-grade reliability (see `reports/phase9b_cidacc_classical_baseline.md`).
- Main failure modes: grid lines, debris false positives, merged touching cells, cluster fragmentation, bright-regime images.
- A deep-learning detector (Phase 9C) is justified **subject to approval**; it is not started here.